# **Customer Churn Analysis for an Hungarian Telecom Company**
**Presented by:**


*   Mukhammad Azim
*   Móger Máté
*   Jose Ocoro

## **Introduction**

The following project is in collaboration with an Hungarian Telecom Company, which in its latest operating periods has noticed a growing churn rate among its customers since the arrival of new competitors to the Hungarian market.

The main objetive of this project is to run an exploratory analyisis from its customer database, understand the reason this churn rate is increasing and develop an ML model to predict potential churn of its customers, in order to prevent these customers from abandoning the company's services through retention campaigns driven by the company's sales and marketing team.

For that purpose, the Data analysis team will work with a dataset of customers provided by the company.



### **1. Loading required packages**


In [5]:
# Core
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt

# Modeling prep
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

ModuleNotFoundError: No module named 'matplotlib'

In [6]:
# Optional - Make outputs easier to read
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 120)

### **2. Loading Dataset**

First submit the file to the Colab environment on the left side taskbar button "Files"

In [ ]:
# Load the dataset
df = pd.read_csv("../data/telco_customer_churn.csv")
df.head()

#### **Explaining Columns**
* **CustomerID:** Unique Customer Identification in the dataset.
* **Gender:** Male or Female
* **SeniorCitizen:** Whether the customer is a senior citizen or not (1, 0)
* **Partner:** Whether the customer has a partner or not (Yes, No)
* **Dependents:** Whether the customer has dependents or not (Yes, No)
* **Tenure:** Number of months the customer has stayed with the company
* **PhoneService:** Whether the customer has a phone service or not (Yes, No)
* **MultipleLines:** Whether the customer has multiple lines or not (Yes, No, No phone service)
* **InternetService:** Customer’s internet service provider (DSL, Fiber optic, No)
* **OnlineSecurity:** Whether the customer has online security or not (Yes, No, No internet service)
* **OnlineBackup:** Whether the customer has online backup or not (Yes, No, No internet service)
* **DeviceProtection:** Whether the customer has device protection or not (Yes, No, No internet service)
* **TechSupport:** Whether the customer has tech support or not (Yes, No, No internet service)
* **StreamingTV:** Whether the customer has streaming TV or not (Yes, No, No internet service)
* **StreamingMovies:** Whether the customer has streaming movies or not (Yes, No, No internet service)
* **Contract:** The contract term of the customer (Month-to-month, One year, Two year)
* **PaperlessBilling:** Whether the customer has paperless billing or not (Yes, No)
* **PaymentMethod:** The customer’s payment method (Electronic check, Mailed check, Bank transfer (automatic), Credit card (automatic))
* **MonthlyCharges:** The amount charged to the customer monthly
* **TotalCharges:** The total amount charged to the customer
* **Churn:** Whether the customer churned or not (Yes or No)




### **3. Dataset Overview + EDA**

#### **Basic Overview**

In [ ]:
print("Shape (rows, cols):", df.shape)
display(df.head(5))

In [ ]:
df.info()

In [ ]:
df.describe(include="all").T

#### **Fix data type issues**

In [ ]:
#TotalCharges
# Convert to numeric
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

In [ ]:
# Check missing values
print("Missing values in TotalCharges:", df["TotalCharges"].isna().sum())

In [ ]:
# Fill missing values with median
df["TotalCharges"] = df["TotalCharges"].fillna(df["TotalCharges"].median())

In [ ]:
# Final check
print(df["TotalCharges"].dtype)
df["TotalCharges"].describe()

In [ ]:
# Convert SeniorCitizen to a proper categorical variable
df["SeniorCitizen"] = df["SeniorCitizen"].map({0: "No", 1: "Yes"})

In [ ]:
df["SeniorCitizen"].value_counts()
df.dtypes

#### **Check Data Quality: Missing Values + Duplicates**

In [ ]:
# Missing Values / NAs
missing = df.isna().sum().sort_values(ascending=False)
missing[missing > 0]

In [ ]:
df.head()

In [ ]:
# Duplicates
dup_count = df.duplicated().sum()
print("Duplicate rows:", dup_count)

No missing values and duplicates on the dataset, that's good news.

#### **Target (Churn) distribution**

In [ ]:
df["Churn"].value_counts(dropna=False)

In [ ]:
churn_rate = (df["Churn"] == "Yes").mean()
print("Baseline churn rate:", round(churn_rate * 100, 2), "%")

26.54% of the customers churned on this dataset

In [ ]:
# Churn Distribution
df["Churn"].value_counts().plot(kind ="bar")
plt.title("Churn Distribution")
plt.xlabel("Churn")
plt.ylabel("Count")
plt.show()

#### **Numeric columns vs Churn**


* Tenure
* MonthlyCharges
* TotalCharges

We have an additional numeric column SeniorCitizen, but it will not be considered in this check because we are going to convert it in a categorical column  with 'Yes' and 'No'.




In [ ]:
numeric_cols = ["tenure", "MonthlyCharges", "TotalCharges"]
df[numeric_cols].describe()

In [ ]:
for col in ["tenure", "MonthlyCharges"]:
    plt.figure(figsize=(6,4))
    df.boxplot(column=col, by="Churn", grid=False)
    plt.title(f"{col} by Churn")
    plt.suptitle("")
    plt.xlabel("Churn")
    plt.ylabel(col)
    plt.tight_layout()
    plt.show()

In [ ]:
num_cols = ["tenure", "MonthlyCharges", "TotalCharges"]

for c in num_cols:
    plt.figure(figsize=(7,4))
    for label in ["No", "Yes"]:
        subset = df[df["Churn"] == label]
        plt.hist(subset[c], bins=30, alpha=0.5, label=label)
    plt.title(f"{c} Distribution by Churn")
    plt.xlabel(c)
    plt.ylabel("Frequency")
    plt.legend(title="Churn")
    plt.tight_layout()
    plt.show()

### **Categorical Variables vs Churn**

In [ ]:
categorical_check_cols = [
    "SeniorCitizen", "Contract", "PaymentMethod",
    "InternetService", "TechSupport", "OnlineSecurity", "PaperlessBilling"
] #Assumed to be the top drivers

In [ ]:
for col in categorical_check_cols:
    print(f"\n--- {col} ---")
    churn_by_group = pd.crosstab(df[col], df["Churn"], normalize="index")
    display(churn_by_group.sort_values("Yes", ascending=False))

Prepare data for model building

In [ ]:
df = df.drop(columns=["customerID"])

In [ ]:
X = df.drop(columns=["Churn"])
y = df["Churn"].map({"No": 0, "Yes": 1})

In [ ]:
y.value_counts()

In [ ]:
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

In [ ]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

In [ ]:
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

Train the first model

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report, RocCurveDisplay
)

In [ ]:
logreg_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000, random_state=42))
])

In [ ]:
logreg_model.fit(X_train, y_train)

In [ ]:
y_pred = logreg_model.predict(X_test)
y_pred_proba = logreg_model.predict_proba(X_test)[:, 1]

In [ ]:
print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("Precision:", round(precision_score(y_test, y_pred), 4))
print("Recall:", round(recall_score(y_test, y_pred), 4))
print("F1-score:", round(f1_score(y_test, y_pred), 4))
print("ROC-AUC:", round(roc_auc_score(y_test, y_pred_proba), 4))

In [ ]:
print(classification_report(y_test, y_pred))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
print(cm)

In [ ]:
RocCurveDisplay.from_predictions(y_test, y_pred_proba)
plt.title("ROC Curve - Logistic Regression")
plt.show()

In [ ]:
feature_names = logreg_model.named_steps["preprocessor"].get_feature_names_out()
coefficients = logreg_model.named_steps["classifier"].coef_[0]

coef_df = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": coefficients
})

coef_df = coef_df.sort_values(by="Coefficient", ascending=False)
coef_df.head(15)

In [ ]:
coef_df.head(10)

In [ ]:
coef_df.tail(10)

In [ ]:
top_positive = coef_df.head(10).sort_values(by="Coefficient")
top_negative = coef_df.tail(10).sort_values(by="Coefficient")

plt.figure(figsize=(8,5))
plt.barh(top_positive["Feature"], top_positive["Coefficient"])
plt.title("Top Features Increasing Churn")
plt.xlabel("Coefficient")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
plt.barh(top_negative["Feature"], top_negative["Coefficient"])
plt.title("Top Features Reducing Churn")
plt.xlabel("Coefficient")
plt.tight_layout()
plt.show()